In [2]:
# TODO:
# 5) upload kernels to Tensorboard after refinement (during refinement)?

# Begin

In [3]:
# @launchit.collected

In [4]:
import os # @launchit.collect
import sys # @launchit.collect
import copy
from collections import namedtuple, defaultdict # @launchit.collect
import json
import datetime
import pprint
from functools import cache
import re
import pickle
from unittest.mock import Mock
import dataclasses # @launchit.collect
from dataclasses import dataclass # @launchit.collect
import IPython
from enum import Flag, StrEnum, auto # @launchit.collect
import multiprocessing as mp

from tqdm.notebook import tqdm

import lark # @launchit.collect

import numpy as np
import einops
import matplotlib.pyplot as plt
import scipy.signal as signal
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import KFold

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim
from torch.utils.data import DataLoader, StackDataset
from torchvision import datasets

import optuna 
from optuna.storages import JournalStorage 
from optuna.storages.journal import JournalFileBackend 
from optuna.trial import TrialState

project_root_path = '${PROJECT_ROOT_PATH}' # @launchit.collect
# @launchit.disable
project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]
# @launchit.stop

sys.path.append(os.path.join(project_root_path, 'lib')) # @launchit.collect
from utils import * # @launchit.collect
from logging_utils import *
from model_registry import *
import launchit # @launchit.disable
import optuna_multiprocessing  # @launchit.collect_2
from metrics_collector import RmqSummaryWriter
from autoincrement import Autoincrement
from torch_helpers import *

# Setup

In [5]:
LOG = Logging.get()
RNG = np.random.default_rng()
METRICS_SUITE = defaultdict(list)

class ExecMode(StrEnum):
    MASTER_NOTEBOOK = auto()
    LAUNCH_NOTEBOOK = auto()
    LAUNCH_MODULE = auto()

CONFIG = namedtuple('Config', 
                    'project_root_path, project_root_uri, subproject_path, data_path, mnist_path, run_path, ' + 
                    'self_fname, self_name, ' +
                    'subproject_name,' +
                    'is_cuda, cuda_device, exec_mode, is_interactive')(
    project_root_path=project_root_path,
    project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
    subproject_path=os.path.abspath('.'),
    data_path=os.path.join(project_root_path, 'data'),
    mnist_path=os.path.join(project_root_path, 'data', 'mnist'),
    run_path='',
    self_fname='',
    self_name='',
    subproject_name='',
    is_cuda=torch.cuda.is_available(),
    cuda_device='cuda' if torch.cuda.is_available() else 'cpu',
    exec_mode=ExecMode.MASTER_NOTEBOOK,
    is_interactive=True,
)

if IPython.get_ipython() is None:
    module_fname = __file__
    module_basename = os.path.basename(module_fname)
    module_name, _ = os.path.splitext(module_basename)
    
    CONFIG = CONFIG._replace(self_fname=module_fname, self_name=module_name)
    CONFIG = CONFIG._replace(exec_mode=ExecMode.LAUNCH_MODULE)
else:
    with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as cf:
        notebook_fname = json.load(cf)['jupyter_session']
        notebook_basename = os.path.basename(notebook_fname)
        notebook_name, notebook_ext = os.path.splitext(notebook_basename)
    
        m = re.match(r'(\w+)-Copy\d+$', notebook_name)
    
        if m: notebook_name = m.group(1) # e.g. Cuml is used to be launched from the copy of the notebook

        CONFIG = CONFIG._replace(self_fname=notebook_fname, self_name=notebook_name)
        
        is_launch = re.match(r'\w+-launch\d+$', notebook_name) is not None
        CONFIG = CONFIG._replace(exec_mode=ExecMode.MASTER_NOTEBOOK if not is_launch else ExecMode.LAUNCH_NOTEBOOK)

CONFIG = CONFIG._replace(is_interactive=CONFIG.exec_mode != ExecMode.LAUNCH_MODULE)

LOG.app_name = CONFIG.self_name
LOG.enable('syslog', not CONFIG.is_interactive)
LOG.enable('stdout', CONFIG.is_interactive)

CONFIG = CONFIG._replace(subproject_name=os.path.basename(os.path.dirname(CONFIG.self_fname)))
CONFIG = CONFIG._replace(run_path=os.path.join(project_root_path, 'run', CONFIG.subproject_name))
LOG(f'CONFIG=\n{pprint.pformat(CONFIG._asdict(), sort_dicts=False)}\n', when=CONFIG.is_interactive)
LOG(f'CONFIG={CONFIG._asdict()}', when=not CONFIG.is_interactive)

os.makedirs(CONFIG.run_path, exist_ok=True)

CONFIG=
{'project_root_path': '/home/misha/dev/mine/neurovision',
 'project_root_uri': 'com.develorium.neurovision',
 'subproject_path': '/home/misha/dev/mine/neurovision/11_cnn',
 'data_path': '/home/misha/dev/mine/neurovision/data',
 'mnist_path': '/home/misha/dev/mine/neurovision/data/mnist',
 'run_path': '/home/misha/dev/mine/neurovision/run/11_cnn',
 'self_fname': '/home/misha/dev/mine/neurovision/11_cnn/s4_cnn_02.ipynb',
 'self_name': 's4_cnn_02',
 'subproject_name': '11_cnn',
 'is_cuda': True,
 'cuda_device': 'cuda',
 'exec_mode': <ExecMode.MASTER_NOTEBOOK: 'master_notebook'>,
 'is_interactive': True}



In [6]:
# @launchit.disable
# @launchit.collect
class LaunchGoal(StrEnum):
    UNSPECIFIED = auto()
    TRAIN_MODEL_CV = auto() # train model and perform cross validation (CV) - model selection stage
    TRAIN_MODEL_AFTER_CV = auto() # we have bunch of models trained via CV, we choose one and retrain it on whole dataset and then calc test_accuracy
    TRAIN_MODEL = auto() # we just want want to train model from scratch and calc test_accuracy (no model selection stage)

FEAT_EXTR_MODEL_TRAIN_GOALS = [LaunchGoal.TRAIN_MODEL_CV, LaunchGoal.TRAIN_MODEL]
TEST_ACCURACY_GOALS = [LaunchGoal.TRAIN_MODEL_AFTER_CV, LaunchGoal.TRAIN_MODEL]

class LaunchGoalObj(namedtuple('LaunchGoalObj', 'goal, model_group_uri, model_name, model_version, model_main_asset_fname')):
    def matches(self, *g):
        if CONFIG.exec_mode == ExecMode.MASTER_NOTEBOOK:
            return True
            
        if g and isinstance(g[0], list):
            assert len(g) == 1
            return self.goal in g[0]

        return self.goal in g

LAUNCH_GOAL = LaunchGoalObj(
    goal=LangUtils.from_str(LaunchGoal, '${LAUNCH_GOAL}', LaunchGoal.UNSPECIFIED),
    model_group_uri='${MODEL_GROUP_URI}',
    model_name='${MODEL_NAME}',
    model_version=LangUtils.from_str(int, '${MODEL_VERSION}', 0),
    model_main_asset_fname='${LAUNCHIT_FNAME}',
)
# @launchit.stop

LAUNCH_GOAL = LAUNCH_GOAL._replace(goal=LaunchGoal.UNSPECIFIED)
LAUNCH_GOAL = LAUNCH_GOAL._replace(model_group_uri=f'{CONFIG.project_root_uri}.{CONFIG.subproject_name}')
LAUNCH_GOAL = LAUNCH_GOAL._replace(model_name=CONFIG.self_name)
LAUNCH_GOAL = LAUNCH_GOAL._replace(model_version=0)
LAUNCH_GOAL = LAUNCH_GOAL._replace(model_main_asset_fname=CONFIG.self_fname)
# @launchit.stop

LOG(f'LAUNCH_GOAL=\n{pprint.pformat(LAUNCH_GOAL._asdict(), sort_dicts=False)}', when=CONFIG.is_interactive)
LOG(f'LAUNCH_GOAL={LAUNCH_GOAL._asdict()}', when=not CONFIG.is_interactive)

LAUNCH_GOAL=
{'goal': <LaunchGoal.UNSPECIFIED: 'unspecified'>,
 'model_group_uri': 'com.develorium.neurovision.11_cnn',
 'model_name': 's4_cnn_02',
 'model_version': 0,
 'model_main_asset_fname': '/home/misha/dev/mine/neurovision/11_cnn/s4_cnn_02.ipynb'}


# Hyperparameters

In [7]:
# @launchit.disable
# @launchit.collect
@dataclass
class Hyperparameters:
    random_seed: int = None
    
    images_preprocessing: str = None
    # model structure params
    fe_model_units: list[str] = None # specs of feature extractor model units, syntax in cells below
    lr_model_units: list[str] = None # specs of LogReg model structure units, syntax in cells below
    # training params
    refine_fe_model: bool = None
    batch_size: int = None
    epochs_count: int = None
    optimizer: str = None
    learn_rate: float | str = None
    cv_folds_count: int = None

    @staticmethod
    def from_dict(d):
        hp = Hyperparameters(**d)
        return hp

    def _asdict(self):
        return dataclasses.asdict(self)

HP = Hyperparameters()
HP.random_seed = 42


In [8]:
@dataclass
class FeatExtrModelUnitParams: 
    Convolution = namedtuple('Convolution', 'in_channels_count out_channels_count in_channels_count_per_kernel kernel_size with_bias')
    Normalization = namedtuple('Normalization', 'norm_method lcn_window_width lcn_window_sigma')
    Pooling = namedtuple('Pooling', 'boxcar_width pool_method kernel_size')
    WeightsSource = namedtuple('WeightsSource', 'asset_name asset_version')
    
    convolution: object = None
    nonlinearity: str = None
    with_gain: bool = None
    rectification: str = None
    normalization: object = None
    pooling: object = None
    weights_source: str = None

@dataclass
class LogRegModelUnitParams: 
    items_count: int = None
    nonlinearity: str = None # any of: None, sigmoid, tanh, ...
    dropout: float = None

@dataclass
class LearnRateParams: 
    Plateau = namedtuple('Plateau', 'factor patience')
    
    learn_rate: float = None
    plateau: object = None

def get_lark_tree_value(tree, var_name, default_value=None):
    try:
        return next(tree.scan_values(lambda i: i.type == var_name)).value
    except StopIteration:
        return default_value

def hp_parse_fe_model_units(self):
    grammar = '''
        spec: conv_spec "->" nonlinearity_spec ("->" gain_spec)? ("->" rectification_spec)? ("->" normalization_spec)? ("->" filtered_pool_spec)? (";" weights_source_spec)?
    
        conv_spec: "conv" "(" IN_CHANNELS_COUNT "->" OUT_CHANNELS_COUNT "(" IN_CHANNELS_COUNT_PER_KERNEL ")" "x" KERNEL_SIZE BIAS_WORD? ")"
        IN_CHANNELS_COUNT: NUMBER
        OUT_CHANNELS_COUNT: NUMBER
        IN_CHANNELS_COUNT_PER_KERNEL: NUMBER
        KERNEL_SIZE: NUMBER
        BIAS_WORD: "+bias" | "-bias"
    
        nonlinearity_spec: NONLINEARITY_WORD
        NONLINEARITY_WORD: WORD
    
        gain_spec: GAIN_WORD
        GAIN_WORD: "gain"
    
        rectification_spec: RECTIFICATION_WORD
        RECTIFICATION_WORD: WORD
    
        normalization_spec: lcn_spec
        lcn_spec: "LCN" "(" LCN_WINDOW_WIDTH "," LCN_WINDOW_SIGMA ")"
        LCN_WINDOW_WIDTH: NUMBER
        LCN_WINDOW_SIGMA: NUMBER
    
        filtered_pool_spec: (boxcar_spec "+")? pool_spec
    
        boxcar_spec: "boxcar(" BOXCAR_WIDTH ")"
        BOXCAR_WIDTH: NUMBER
        
        pool_spec: POOL_METHOD "(" POOL_KERNEL_SIZE ")"
        POOL_METHOD: "avg" | "max"
        POOL_KERNEL_SIZE: NUMBER

        weights_source_spec: "from:" ASSET_NAME "," ASSET_VERSION
        ASSET_NAME: IDENTIFIER
        ASSET_VERSION: IDENTIFIER

        IDENTIFIER: ("a".."z"|"A".."Z"|"0".."9"|"_"|"-")+
        
        %import common.WORD
        %import common.NUMBER
        %import common.WS
        %ignore WS
    '''
    parser = lark.Lark(grammar, start='spec')
    params_list = []

    for unit in self.fe_model_units:
        tree = parser.parse(unit)
        gtv = lambda var_name, default_value='': get_lark_tree_value(tree, var_name, default_value)
        params = FeatExtrModelUnitParams()
        params.convolution = FeatExtrModelUnitParams.Convolution(
            in_channels_count=int(gtv('IN_CHANNELS_COUNT')),
            out_channels_count=int(gtv('OUT_CHANNELS_COUNT')),
            in_channels_count_per_kernel=int(gtv('IN_CHANNELS_COUNT_PER_KERNEL')),
            kernel_size=int(gtv('KERNEL_SIZE')),
            with_bias=gtv('BIAS_WORD', '') == "+bias"
        )
    
        params.nonlinearity = gtv('NONLINEARITY_WORD')
        params.with_gain = bool(gtv('GAIN_WORD', ''))
        params.rectification = gtv('RECTIFICATION_WORD', None)
        
        if list(tree.find_data('lcn_spec')):
            params.normalization = FeatExtrModelUnitParams.Normalization(
                norm_method='LCN', 
                lcn_window_width=int(gtv('LCN_WINDOW_WIDTH')),
                lcn_window_sigma=int(gtv('LCN_WINDOW_SIGMA')),
        )
    
        params.pooling = FeatExtrModelUnitParams.Pooling(
            boxcar_width=int(gtv('BOXCAR_WIDTH', 0)),
            pool_method=gtv('POOL_METHOD'),
            kernel_size=int(gtv('POOL_KERNEL_SIZE')),
        )
        
        if list(tree.find_data('weights_source_spec')):
            params.weights_source = FeatExtrModelUnitParams.WeightsSource(
                asset_name=gtv('ASSET_NAME'), 
                asset_version=gtv('ASSET_VERSION'), 
        )
            
        params_list.append(params)

    return params_list

def hp_parse_lr_model_units(self):
    grammar = '''
        spec: (dropout_spec "->")? items_count_spec ("->" nonlinearity_spec)?
    
        dropout_spec: "dropout" "(" DROPOUT_P ")"
        DROPOUT_P: NUMBER
    
        items_count_spec: ITEMS_COUNT
        ITEMS_COUNT: NUMBER
    
        nonlinearity_spec: NONLINEARITY_WORD
        NONLINEARITY_WORD: WORD
    
        %import common.WORD
        %import common.NUMBER
        %import common.WS
        %ignore WS
    '''
    parser = lark.Lark(grammar, start='spec')
    params_list = []

    for unit in self.lr_model_units:
        tree = parser.parse(unit)
        gtv = lambda var_name, default_value='': get_lark_tree_value(tree, var_name, default_value)
        params = LogRegModelUnitParams()
        params.items_count = int(gtv('ITEMS_COUNT'))
        params.nonlinearity = gtv('NONLINEARITY_WORD')
        params.dropout = float(gtv('DROPOUT_P')) if gtv('DROPOUT_P', False) else None
        params_list.append(params)

    return params_list

def hp_parse_learn_rate(self):
    params = LearnRateParams()

    if isinstance(self.learn_rate, float):
        params.learn_rate = self.learn_rate
        return params
        
    grammar = '''
        spec: initial_lr_spec(","plateau_spec)?
    
        initial_lr_spec: INITIAL_LR
        INITIAL_LR: NUMBER
    
        plateau_spec: "plateau" "(" (|plateau_params_spec ("," plateau_params_spec)*) ")"
        plateau_params_spec: plateau_factor_spec | plateau_patience_spec
        plateau_factor_spec: "factor" "=" plateau_factor_value_spec
        plateau_factor_value_spec: PLATEAU_FACTOR_VALUE
        plateau_patience_spec: "patience" "=" plateau_patience_value_spec
        plateau_patience_value_spec: PLATEAU_PATIENCE_VALUE
        PLATEAU_FACTOR_VALUE: NUMBER
        PLATEAU_PATIENCE_VALUE: NUMBER
        
        %import common.NUMBER
        %import common.WS
        %ignore WS
    '''
    parser = lark.Lark(grammar, start='spec')
    tree = parser.parse(self.learn_rate)
    gtv = lambda var_name, default_value='': get_lark_tree_value(tree, var_name, default_value)
    params.learn_rate = float(gtv('INITIAL_LR'))
    
    if list(tree.find_data('plateau_spec')):
        params.plateau = LearnRateParams.Plateau(
            factor=float(gtv('PLATEAU_FACTOR_VALUE', 0.1)),
            patience=int(gtv('PLATEAU_PATIENCE_VALUE', 10)),
        )

    return params

Hyperparameters.parse_fe_model_units = hp_parse_fe_model_units
Hyperparameters.parse_lr_model_units = hp_parse_lr_model_units
Hyperparameters.parse_learn_rate = hp_parse_learn_rate

In [9]:
# @launchit.disable
x = Hyperparameters(fe_model_units=[
    'conv(1->64(1)x9+bias)->tanh->gain->abs->LCN(9,2)->boxcar(9)+avg(2); from:s4_psd_01,397',
    'conv(64->32(1)x9)->tanh->max(2)',
])
x.parse_fe_model_units()

[FeatExtrModelUnitParams(convolution=Convolution(in_channels_count=1, out_channels_count=64, in_channels_count_per_kernel=1, kernel_size=9, with_bias=True), nonlinearity='tanh', with_gain=True, rectification='abs', normalization=Normalization(norm_method='LCN', lcn_window_width=9, lcn_window_sigma=2), pooling=Pooling(boxcar_width=9, pool_method='avg', kernel_size=2), weights_source=WeightsSource(asset_name='s4_psd_01', asset_version='397')),
 FeatExtrModelUnitParams(convolution=Convolution(in_channels_count=64, out_channels_count=32, in_channels_count_per_kernel=1, kernel_size=9, with_bias=False), nonlinearity='tanh', with_gain=False, rectification=None, normalization=None, pooling=Pooling(boxcar_width=0, pool_method='max', kernel_size=2), weights_source=None)]

In [10]:
# @launchit.disable
x = Hyperparameters(lr_model_units=[
    'dropout(0.5)->200->tanh',
    '200->sigmoid',
])
x.parse_lr_model_units()

[LogRegModelUnitParams(items_count=200, nonlinearity='tanh', dropout=0.5),
 LogRegModelUnitParams(items_count=200, nonlinearity='sigmoid', dropout=None)]

In [11]:
# @launchit.disable
x = Hyperparameters(learn_rate='0.005,plateau(factor=0.115, patience=15)')
lr_params = x.parse_learn_rate()
o = torch.optim.Adam(params=[nn.Parameter(torch.tensor(1.))])
p = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=o, **lr_params.plateau._asdict())
lr_params, p.factor, p.patience

(LearnRateParams(learn_rate=0.005, plateau=Plateau(factor=0.115, patience=15)),
 0.115,
 15)

# Launch

## new_model_registry

In [12]:
def new_model_registry(is_real=None):
    is_launch = CONFIG.exec_mode in [ExecMode.LAUNCH_NOTEBOOK, ExecMode.LAUNCH_MODULE]
    is_real = is_real if is_real is not None else is_launch

    if not is_real:
        mr = Mock()
        mr.register_model.return_value = 0
        return mr
        
    return ModelRegistry(LAUNCH_GOAL.model_group_uri)

## new_summary_writer

In [13]:
def new_summary_writer(log_dir, is_real=None):
    is_launch = CONFIG.exec_mode in [ExecMode.LAUNCH_NOTEBOOK, ExecMode.LAUNCH_MODULE]
    is_real = is_real if is_real is not None else is_launch

    if not is_real:
        sw = Mock()
        sw.flush.side_effect = sw.reset_mock # to get rid of all recorded call_args_list, which might be heavy (e.g. add_figure)
        return sw
    
    return RmqSummaryWriter(log_dir)

## Bootstrap

In [14]:
optuna_trial = optuna_multiprocessing.get_trial()
optuna_trial_subdir_name = ''

if optuna_trial is not None:
    optuna_trial.set_user_attr('MODEL_VERSION', LAUNCH_GOAL.model_version)
    study_serial = optuna_trial.user_attrs['STUDY_SERIAL']
    optuna_trial_subdir_name = f'opt_{study_serial}'
    LOG(f'Optuna {optuna_trial.number=}, {optuna_trial.user_attrs=}')

LOG(f'HP={HP._asdict()}', when=not CONFIG.is_interactive)
    
if HP.random_seed is not None:
    torch.manual_seed(HP.random_seed)
    RNG = np.random.default_rng(HP.random_seed)    
    LOG(f'Random seed={HP.random_seed}')

if LAUNCH_GOAL.goal != LaunchGoal.TRAIN_MODEL_AFTER_CV:
    model_registry = new_model_registry()
    model_registry.attach_asset(LAUNCH_GOAL.model_name, LAUNCH_GOAL.model_version, CONFIG.self_fname, replace=True)
        
    meta = dict(
        optuna_trial_number=getattr(optuna_trial, 'number', None),
        hypers=HP._asdict(), 
        config=CONFIG._asdict(), 
    )
    
    with io.StringIO() as b:
        json.dump(meta, b)
        model_registry.attach_asset(LAUNCH_GOAL.model_name, LAUNCH_GOAL.model_version, b, asset_ext='json', asset_classifier='meta', replace=True)

summary_log_dir = LAUNCH_GOAL.model_name
summary_log_dir = os.path.join(summary_log_dir, optuna_trial_subdir_name) if optuna_trial_subdir_name != '' else summary_log_dir 
summary_log_dir = os.path.join(summary_log_dir, str(LAUNCH_GOAL.model_version))
LOG(f'Tensorboard run={summary_log_dir}')
summary_writer = new_summary_writer(log_dir=summary_log_dir)
summary_writer.add_text('hypers', pprint.pformat(HP._asdict(), sort_dicts=False), 1)
summary_writer.add_text('config', pprint.pformat(CONFIG._asdict(), sort_dicts=False), 1)

Random seed=42
Tensorboard run=s4_cnn_02/0


<Mock name='mock.add_text()' id='139604438735888'>

# Images

In [15]:
# @launchit.disable
# @launchit.collect_1
HP.images_preprocessing = 'SAMPLE_WISE' # NONE, UNINORM, SAMPLE_WISE, MIN_MAX, STANDARDIZE, ZCA_WHITEN, ZCA_HFR30_WHITEN

## get_mnist_images

In [16]:
def get_mnist_images(subdataset='TRAIN'):
    assert subdataset in ['TRAIN', 'TEST'], f'Unsupported subdataset={subdataset}'
    d = datasets.MNIST(CONFIG.mnist_path, train=subdataset=='TRAIN', download=True)
    images = d.data.numpy()
    images = images.astype('float32')
    image_labels = d.targets
    return images, image_labels

## UninormScaler

In [17]:
class UninormScaler:
    def __init__(self, divisor=255.0):
        self.divisor = divisor
        
    def fit_transform(self, images):
        return self.transform(images)

    def transform(self, images):
        return images / self.divisor

## SampleWiseScaler

In [18]:
# https://gist.github.com/kocherovms/ca352c30fe3eea0f155d4862ddde6e3a for tests and breakdown
class SampleWiseScaler:
    def fit_transform(self, images):
        return self.transform(images)

    # Images are expected to be in raveled (flattened) mode => only last dim is taken into account
    def transform(self, images):
        shape = images.shape
        images = images.reshape(-1, images.shape[-1]) # get rid of all dimensions except the last one
        means = images.mean(axis=-1)
        stds = images.std(axis=-1)
        images = images.T - means
        images = images / np.where(stds != 0, stds, 1)
        images = images.T
        return images.reshape(shape)

## preprocess_images

In [19]:
# images are expected to be in raveled (flattened) mode
def preprocess_images(images, preprocessing_method, scaler=None):
    match preprocessing_method:
        case 'UNINORM':
            scaler = UninormScaler() if scaler is None else scaler
            images = scaler.fit_transform(images)
        case 'SAMPLE_WISE':
            scaler = SampleWiseScaler() if scaler is None else scaler
            images = scaler.fit_transform(images)
        case 'MIN_MAX':
            scaler = MinMaxScaler() if scaler is None else scaler
            images = scaler.fit_transform(images)
        case 'STANDARDIZE':
            scaler = StandardScaler() if scaler is None else scaler
            images = scaler.fit_transform(images)
        case 'ZCA_WHITEN':
            scaler = StandardScaler(with_std=False) if scaler is None else scaler
            images = scaler.fit_transform(images)
            
            Σ = np.cov(images, rowvar=False)
            u, s, _ = np.linalg.svd(Σ)
            images = (u @ np.diag(1.0 / np.sqrt(s + 1e-6)) @ u.T @ images.T).T
        case 'ZCA_HFR30_WHITEN': # HFR30 - Remove 30% of High Frequencies
            scaler = StandardScaler(with_std=False) if scaler is None else scaler
            images = scaler.fit_transform(images)
    
            Σ = np.cov(images, rowvar=False)
            eigvals, eigvecs = np.linalg.eig(Σ)
            eigvals_order = np.argsort(-eigvals)
            wipeout_inds = eigvals_order[int(len(eigvals_order) * (1 - 0.3)):]
            eigvals_w = eigvals.copy()
            eigvals_w[wipeout_inds] = 0
            
            R, S = eigvecs, np.diag(np.sqrt(eigvals_w)) # R - rotation matrix, S - scale matrix
            S_inv = np.reciprocal(S, out=np.zeros_like(S), where=(S != 0))
            R_inv = R.T
            W = R @ S_inv @ R_inv  # equiv. to: R @ np.eye(len(S_inv)) @ S_inv @ R_inv
            images = (W @ images.T).T
        case 'NONE':
            pass
        case _:
            assert False, f'Unsupported preprocessing_method={preprocessing_method}'

    return images, scaler

# Dataset

## create_dataset_for_supervised_training

In [20]:
def create_dataset_for_supervised_training(name, is_raveled=False, scaler=None):
    images, labels = get_mnist_images(name)
    shape = images.shape
    images = images.reshape(len(images), -1)
    images, scaler = preprocess_images(images, HP.images_preprocessing, scaler)
    images = images if is_raveled else images.reshape(shape)
    images = torch.Tensor(images)
    images = images.contiguous() # force dense memory layout (speeds up DataLoader x2 in case of any transposes within preprocess_images)
    dataset = StackDataset(images, labels)
    return dataset, scaler

## create_dataset_couple_for_supervised_training

In [21]:
def create_dataset_couple_for_supervised_training(is_raveled=False):
    train_dataset, scaler = create_dataset_for_supervised_training('TRAIN', is_raveled=is_raveled, scaler=None)
    test_dataset, _ = create_dataset_for_supervised_training('TEST', is_raveled=is_raveled, scaler=scaler)
    return train_dataset, test_dataset

# Model

## FeatExtrModelUnit

In [22]:
class FeatExtrModelUnit(nn.Module):
    def __init__(self, unit_hp):
        super().__init__()
        convolution = unit_hp.convolution
        groups_count = convolution.in_channels_count // convolution.in_channels_count_per_kernel
        self.conv = nn.Conv2d(
            in_channels=convolution.in_channels_count, 
            out_channels=convolution.out_channels_count, 
            kernel_size=convolution.kernel_size, 
            padding='valid',
            bias=convolution.with_bias,
            groups=groups_count
        )

        if unit_hp.nonlinearity is None:
            self.nonlinearity = lambda i: i
        else:
            self.nonlinearity = getattr(F, unit_hp.nonlinearity)

        if not unit_hp.with_gain:
            self.gain = lambda i: i
        else:
            self.gain_params = nn.Parameter(torch.ones(1, 1, convolution.out_channels_count)) 
            self.gain = self.apply_gain

        if (rectification := unit_hp.rectification) is None:
            self.rectification = lambda i: i
        else:
            match rectification:
                case 'abs':
                    self.rectification = lambda i: i.abs()
                case 'relu':
                    self.rectification = F.relu
                case None:
                    self.rectification = lambda i: i
                case _:
                    assert False, f'Unsupported {rectification=}'

        if (normalization := unit_hp.normalization) is None:
            self.normalization = lambda i: i
        else:
            match normalization.norm_method:
                case 'LCN':
                    # There may be problems when moving model between GPU and CPU due to lambda closure side effects
                    self.normalization = lambda i: self.apply_lcn(i, normalization.lcn_window_width, normalization.lcn_window_sigma)
                case None:
                    self.normalization = lambda i: i
                case _:
                    assert False, f'Unsupported {normalization.norm_method=}'

        pooling = unit_hp.pooling

        if pooling.boxcar_width is None or pooling.boxcar_width == 0:
            self.boxcar = lambda i: i
        else:
            assert pooling.boxcar_width > 0
            padding = pooling.boxcar_width // 2 
            self.boxcar = nn.AvgPool2d(kernel_size=pooling.boxcar_width, stride=1, padding=padding, count_include_pad=False)

        match pooling.pool_method:
            case 'avg':
                self.pool = nn.AvgPool2d(kernel_size=pooling.kernel_size)
            case 'max':
                self.pool = nn.MaxPool2d(kernel_size=pooling.kernel_size)
            case _:
                assert False, f'Unsupported {pooling.pool_method=}'

        if unit_hp.weights_source is not None:
            self.load_weights(unit_hp.weights_source)

    def forward(self, inp): 
        # inp.shape: batch, channel, height, width
        res = self.conv(inp)
        res = self.nonlinearity(res)
        res = self.gain(res)
        res = self.rectification(res)
        res = self.normalization(res)
        res = self.boxcar(res)
        res = self.pool(res)
        return res

    def apply_gain(self, inp):
        # See https://gist.github.com/kocherovms/1568d303d584d5033af3c169b99eaa33 for breakdown of gain multiplication logic
        res = inp
        shape = res.shape
        res = res.reshape(res.shape[0], res.shape[1], -1)
        res = res.transpose(1, 2)
        res = self.gain_params * res
        res = res.transpose(1, 2)
        res = res.reshape(shape)
        return res

    def apply_lcn(self, feature_maps, window_width, window_sigma):
        # feature_maps.shape: batch, feature_map, height, width
        # See https://gist.github.com/kocherovms/a5a39939dafcee21e0754bad043adeca for breakdown of LCN logic
        if not hasattr(self, 'lcn_conv') or self.lcn_conv is None or self.lcn_conv.in_channels != feature_maps.shape[1] or self.lcn_conv.weight.device != feature_maps.device:
            window = signal.windows.gaussian(window_width, window_sigma)[:,np.newaxis] 
            window = window @ window.T # aka 2d kernel
            window_norm = window / window.sum() / feature_maps.shape[1]
            self.lcn_conv = nn.Conv2d(
                in_channels=feature_maps.shape[1], 
                out_channels=feature_maps.shape[1],
                kernel_size=window_width,
                padding='same',
                bias=False,
                groups=feature_maps.shape[1], # one filter per feature_map
                device=feature_maps.device,
            )
            self.lcn_conv.requires_grad_(False)
            self.lcn_conv.weight[:,:,...] = torch.tensor(window_norm).to(device=CONFIG.cuda_device) # upload window to conv kernels

        convolved = self.lcn_conv(feature_maps)
        means = convolved.sum(dim=-3)
        means = einops.rearrange(means, 'b h w -> b 1 h w')
        v = feature_maps - means
        squared_v = v ** 2
        convolved = self.lcn_conv(squared_v)
        sigmas = torch.sqrt(convolved.sum(dim=-3))
        c = einops.rearrange(sigmas, 'b h w -> b (h w)').mean(axis=-1)
        sigmas = einops.rearrange(sigmas, 'b h w -> b 1 h w')
        c = einops.rearrange(c, 'b -> b 1 1 1')
        return v / torch.where(sigmas > c, sigmas, c)

    def load_weights(self, weights_source):
        mr = new_model_registry(is_real=True)

        for classifier, target, load_func in zip(
            ('kernels', 'biases', 'gains'), 
            (self.conv.weight, self.conv.bias, self.gain_params),
            (self.load_kernels, self.load_biases, self.load_gain_params),
        ):
            if target is None:
                continue

            # LOG(weights_source.asset_name, weights_source.asset_version)
            asset = mr.get_asset_content(weights_source.asset_name, weights_source.asset_version, 'npy', classifier)

            with torch.no_grad():
                with io.BytesIO(asset) as b:
                    asset = np.load(b)
                    asset = torch.tensor(asset)
                    # LOG('Before', classifier, asset.shape, target.shape, asset.sum(), target.sum())
                    sum = load_func(asset)
                    # LOG('After', asset.sum(), target.sum())
                    assert asset.sum() == target.sum()
                    LOG(f'{classifier} loaded from {weights_source.asset_name}:{weights_source.asset_version}')

    def load_kernels(self, asset):
        shape = einops.parse_shape(self.conv.weight, 'out_ch in_ch h w')
        self.conv.weight[:,:,...] = einops.rearrange(asset, 'k (h w) -> k 1 h w', h=shape['h'], w=shape['w'])

    def load_biases(self, asset):
        self.conv.bias[:] = asset

    def load_gain_params(self, asset):
        self.gain_params.data[:,:,...] = einops.rearrange(asset, 'g -> 1 1 g')

## FeatExtrModel

In [23]:
class FeatExtrModel(nn.Module):
    def __init__(self, model_hp):
        super().__init__()
        self.units = nn.Sequential()

        for unit_hp in model_hp.parse_fe_model_units():
            unit = FeatExtrModelUnit(unit_hp)
            self.units.append(unit)

    def forward(self, inp):
        if self.training:
            return self.units(inp)
        else:
            with torch.no_grad():
                return self.units(inp)

In [24]:
# @launchit.disable
model_hp = Hyperparameters(fe_model_units=[
    'conv(1->32(1)x5+bias)->tanh->gain->abs->LCN(9,2)->avg(2); from:s4_psd_01,396',
    'conv(32->64(16)x5)->tanh->abs->boxcar(9)+avg(2)',
])
model = FeatExtrModel(model_hp)
sum([p.numel() for p in model.parameters()]), model

kernels loaded from s4_psd_01:396
biases loaded from s4_psd_01:396
gains loaded from s4_psd_01:396


(26464,
 FeatExtrModel(
   (units): Sequential(
     (0): FeatExtrModelUnit(
       (conv): Conv2d(1, 32, kernel_size=(5, 5), stride=(1, 1), padding=valid)
       (pool): AvgPool2d(kernel_size=2, stride=2, padding=0)
     )
     (1): FeatExtrModelUnit(
       (conv): Conv2d(32, 64, kernel_size=(5, 5), stride=(1, 1), padding=valid, groups=2, bias=False)
       (boxcar): AvgPool2d(kernel_size=9, stride=1, padding=4)
       (pool): AvgPool2d(kernel_size=2, stride=2, padding=0)
     )
   )
 ))

## LogRegModelUnit

In [25]:
class LogRegModelUnit(nn.Module):
    def __init__(self, input_dims_count, unit_hp):
        super().__init__()
        self.linear = nn.Linear(
            in_features=input_dims_count, 
            out_features=unit_hp.items_count, 
            bias=True
        )

        if unit_hp.nonlinearity is None:
            self.nonlinearity = lambda i: i
        else:
            self.nonlinearity = getattr(F, unit_hp.nonlinearity)

        if unit_hp.dropout is None:
            self.dropout = lambda i: i
        else:
            assert unit_hp.dropout >= 0
            self.dropout = nn.Dropout(unit_hp.dropout)

    def forward(self, inp): 
        res = self.dropout(inp)
        res = self.linear(res)
        res = self.nonlinearity(res)
        return res

## MainModel

In [26]:
class MainModel(nn.Module):
    def __init__(self, image_shape, fe_model, model_hp):
        super().__init__()
        self.pipeline = nn.Sequential()
        self.pipeline.append(fe_model)
        self.pipeline.append(nn.Flatten(1))

        with torch.no_grad():
            probe_tensor = torch.ones(1, 1, *image_shape)
            result_tensor = fe_model(probe_tensor)
            input_dims_count = result_tensor.reshape(1, -1).shape[-1]

        for unit_hp in model_hp.parse_lr_model_units():
            unit = LogRegModelUnit(input_dims_count, unit_hp)
            input_dims_count = unit_hp.items_count
            self.pipeline.append(unit)

    def forward(self, inp):
        return self.pipeline(inp)

In [27]:
# @launchit.disable
fe_model_hp = Hyperparameters(fe_model_units=[
    'conv(1->32(1)x5+bias)->tanh->gain->abs->LCN(9,2)->avg(2)',
    'conv(32->64(16)x5)->tanh->abs->boxcar(9)+avg(2)',
])
fe_model = FeatExtrModel(fe_model_hp)

model_hp = Hyperparameters(lr_model_units=[
    'dropout(0.5)->200->tanh',
    'dropout(0.2)->200->tanh',
    '10->tanh',
])
model = MainModel((28, 28), fe_model, model_hp)
sum([p.numel() for p in model.parameters()])
probe_tensor = torch.ones(1, 1, 28, 28)
model(probe_tensor)

tensor([[-0.2452, -0.0284,  0.1871, -0.0099, -0.0957, -0.0273,  0.0128, -0.0453,
         -0.0431,  0.0586]], grad_fn=<TanhBackward0>)

# Model training via CV

Build number of model instances, train each on separate cross-validation fold from training dataset => train and validation metrics

## Configure

In [27]:
# @launchit.disable
# @launchit.collect_1
HP.fe_model_units = [
    'conv(1->32(1)x5+bias)->tanh->gain->abs->avg(2); from:s4_psd_01,450',
    'conv(32->64(16)x5+bias)->tanh->gain->abs->avg(2)'
]
HP.lr_model_units = [
    '200->tanh',
    '10->tanh',
]
HP.refine_fe_model = True
HP.batch_size = 100
HP.epochs_count = 1
HP.optimizer = 'Adam'
HP.learn_rate = 0.001
HP.cv_folds_count = 5
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'random_seed': 42,
 'images_preprocessing': 'SAMPLE_WISE',
 'fe_model_units': ['conv(1->32(1)x5+bias)->tanh->gain->abs->avg(2); '
                    'from:04_psd_01,396',
                    'conv(32->64(16)x5+bias)->tanh->gain->abs->avg(2)'],
 'lr_model_units': ['200->tanh', '10->tanh'],
 'refine_fe_model': True,
 'batch_size': 100,
 'epochs_count': 1,
 'optimizer': 'Adam',
 'learn_rate': 0.001,
 'cv_folds_count': 5}


## Train

In [126]:
# @launchit.disable_1
if LAUNCH_GOAL.matches(LaunchGoal.TRAIN_MODEL_CV):
    fe_model = FeatExtrModel(HP)
    fe_model = fe_model.to(device=CONFIG.cuda_device)
    k_fold = KFold(HP.cv_folds_count, shuffle=True, random_state=HP.random_seed)
    metrics_suite = defaultdict(list)
    optuna_trial = optuna_multiprocessing.get_trial()
    dataset, _ = create_dataset_for_supervised_training('TRAIN')
    
    for fold_ind, (train_inds, val_inds) in tqdm(enumerate(k_fold.split(dataset)), desc='Fold', total=HP.cv_folds_count, disable=not CONFIG.is_interactive):
        with LOG.auto_prefix('FOLD', fold_ind):
            data_loader = DataLoader(StackDataset(*dataset[train_inds]), batch_size=HP.batch_size, shuffle=True)
            val_images = dataset[val_inds][0].to(device=CONFIG.cuda_device)
            val_images = einops.rearrange(val_images, 'b h w -> b 1 h w')
            val_labels = dataset[val_inds][1].to(device=CONFIG.cuda_device)
    
            if HP.refine_fe_model:
                this_fe_model = copy.deepcopy(fe_model)
                this_fe_model.train()
            else:
                this_fe_model = fe_model
                this_fe_model.eval()
                
            model = MainModel(dataset[0][0].shape, this_fe_model.to(device='cpu'), HP)
            model = model.to(device=CONFIG.cuda_device)
            loss_fn = nn.CrossEntropyLoss()
            lr_params = HP.parse_learn_rate()
            optimizer = getattr(torch.optim, HP.optimizer)(model.parameters(), lr=lr_params.learn_rate)
            lr_scheduler = LrSchedulerWrapper(optimizer, lr_params)
            fold_metrics_suite = defaultdict(list)
        
            for epoch_ind in tqdm(range(HP.epochs_count), 'Epoch', disable=not CONFIG.is_interactive):
                epoch = epoch_ind + 1
                
                with LOG.auto_prefix('EPOCH', epoch):
                    train_loss_sum, train_loss_denom = 0, 0
                    train_accuracy_sum, train_accuracy_denom = 0, 0
                    
                    for batch in data_loader:
                        optimizer.zero_grad()
                        
                        images = batch[0].to(device=CONFIG.cuda_device)
                        images = einops.rearrange(images, 'b h w -> b 1 h w')
                        labels = batch[1].to(device=CONFIG.cuda_device)
                        logits = model(images)
                        loss = loss_fn(logits, labels)
                        loss.backward()
                        
                        optimizer.step()
                        
                        train_loss_sum += loss.item() * len(batch)
                        train_loss_denom += len(batch)
                        train_accuracy_sum += (logits.argmax(axis=1) == labels).sum().item() # how many correct predictions
                        train_accuracy_denom += len(labels)
                
                    assert train_loss_denom > 0
                    assert train_accuracy_denom > 0
                    train_loss = train_loss_sum / train_loss_denom
                    train_accuracy = train_accuracy_sum / train_accuracy_denom
                    fold_metrics_suite['train_loss'].append(train_loss)
                    fold_metrics_suite['train_accuracy'].append(train_accuracy)
                    
                    lr_scheduler.step(train_loss)
                
                    with torch.no_grad():
                        val_logits = model(val_images)
                        val_loss = loss_fn(val_logits, val_labels).item()
                        predicted_val_labels = val_logits.argmax(axis=1)
                        val_accuracy = (predicted_val_labels == val_labels).sum().item() / len(val_labels)
                        fold_metrics_suite['val_loss'].append(val_loss)
                        fold_metrics_suite['val_accuracy'].append(val_accuracy)
    
                    LOG(f'{train_loss=:.2f}, {train_accuracy=:.2f}, {val_loss=:.2f}, {val_accuracy=:.2f}', when=not CONFIG.is_interactive)
                    
                    if optuna_trial is not None and fold_ind == 0:
                        # https://optuna.readthedocs.io/en/stable/reference/generated/optuna.trial.Trial.html#optuna.trial.Trial.report:
                        # > If this method is called multiple times at the same step in a trial, the reported value only the first time is stored 
                        # > and the reported values from the second time are ignored.
                        # In other words calling report for fold other than the first one does nothing except producing tons of warnings in console. 
                        # As such only the first fold is indicative
                        optuna_trial.report(val_accuracy, epoch_ind) # "Note that pruners assume that step starts at zero" -> as such use epoch_ind instead of epoch
                
                        if optuna_trial.should_prune():
                            # Despite written in docs OPTUNA_TRIAL.should_prune is not idempotent - consequent calls could lead
                            # to different responses. Perhapse this is due to influence of concurrent trials running which could change
                            # pruner decision. As such cache pruning result so it's immutable
                            optuna_trial.set_user_attr('IS_PRUNED', True)
                            break
        
            for metric_name, metric_values in fold_metrics_suite.items():
                metrics_suite[metric_name].append(metric_values)
        
            if 'IS_PRUNED' in getattr(optuna_trial, 'user_attrs', {}):
                LOG(f'Optuna pruning condition encountered. Stopping training')
                break
        
    for metric_name in list(metrics_suite.keys()):
        metric_values = metrics_suite[metric_name]
        metric_values = np.array(metric_values)
        # According to Optuna docs we could get pruned only on first fold (report() calls is ignored on successive folds).
        # As such we either should have 1) a full-fledged metrcs_suite (for all folds and for each epoch)
        # or 2) only metrics from the very first fold
        if 'IS_PRUNED' in getattr(optuna_trial, 'user_attrs', {}):
            assert len(metric_values) == 1
            assert metric_values.shape[1] <= HP.epochs_count
        else:
            assert len(metric_values) == HP.cv_folds_count
            assert metric_values.shape[1] == HP.epochs_count
            
        mean_values = list(metric_values.mean(axis=0)) # coercing to list since json.dump throws "np.ndarray is not serializable"
        std_values = list(metric_values.std(axis=0)) # coercing to list since json.dump throws "np.ndarray is not serializable"
        metrics_suite['mean_' + metric_name] = mean_values
        metrics_suite['std_' + metric_name] = std_values
        assert len(mean_values) == len(std_values)
        
        for epoch_ind in range(min(HP.epochs_count, len(mean_values))):
            epoch = epoch_ind + 1
            summary_writer.add_scalar('mean_' + metric_name, mean_values[epoch_ind], epoch)
            summary_writer.add_scalar('std_' + metric_name, std_values[epoch_ind], epoch)
            
        summary_writer.flush()   
    
    METRICS_SUITE.update(metrics_suite)

Fold:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

## Save

In [165]:
# @launchit.disable_1
if LAUNCH_GOAL.matches(LaunchGoal.TRAIN_MODEL_CV):
    model_registry = new_model_registry()
    
    with io.BytesIO() as b:
        # Only model from last KFold is saved. A bit senseless...
        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'hypers': HP._asdict(),
        }, b)
        model_registry.attach_asset(LAUNCH_GOAL.model_name, LAUNCH_GOAL.model_version, b, asset_ext='pt', asset_classifier='cv', replace=True)
    
    with io.StringIO() as b:
        json.dump(METRICS_SUITE, b)
        model_registry.attach_asset(LAUNCH_GOAL.model_name, LAUNCH_GOAL.model_version, b, asset_ext='json', asset_classifier='metrics', replace=True)

# Model training (test_accuracy)

Build model, train on whole train dataset (without cross validation), validate versus test dataset => test_accuracy.

## Configure

In [104]:
# @launchit.disable
# @launchit.collect_1
HP.fe_model_units = [
    'conv(1->32(1)x5+bias)->tanh->gain->abs->LCN(5,1)->avg(2)',
    'conv(32->64(16)x5+bias)->tanh->gain->abs->avg(2)'
]
HP.lr_model_units = [
    '200->tanh',
    '10->tanh',
]
HP.refine_fe_model = True
HP.batch_size = 100
HP.epochs_count = 1
HP.optimizer = 'Adam'
HP.learn_rate = 0.001
HP.cv_folds_count = 5
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'random_seed': 42,
 'images_preprocessing': 'SAMPLE_WISE',
 'fe_model_units': ['conv(1->32(1)x5+bias)->tanh->gain->abs->LCN(5,1)->avg(2)',
                    'conv(32->64(16)x5+bias)->tanh->gain->abs->avg(2)'],
 'lr_model_units': ['200->tanh', '10->tanh'],
 'refine_fe_model': True,
 'batch_size': 100,
 'epochs_count': 1,
 'optimizer': 'Adam',
 'learn_rate': 0.001,
 'cv_folds_count': 5}


## Load

In [105]:
# @launchit.disable_2
if CONFIG.exec_mode == ExecMode.MASTER_NOTEBOOK:
    fe_model = FeatExtrModel(HP)
    fe_model = fe_model.to(device=CONFIG.cuda_device)
elif LAUNCH_GOAL.matches(LaunchGoal.TRAIN_MODEL):
    fe_model = FeatExtrModel(HP)
    fe_model = fe_model.to(device=CONFIG.cuda_device)
elif LAUNCH_GOAL.matches(TEST_ACCURACY_GOALS):
    model_registry = new_model_registry()
    meta = json.load(io.BytesIO(model_registry.get_asset_content(LAUNCH_GOAL.model_name, LAUNCH_GOAL.model_version, asset_ext='json', asset_classifier='meta')))
    # This launch should EXTEND model storage on Nexus with artifact of trained model on whole dataset. 
    # We lack proper hyperparameters from this launch notebook per se (see corresponding launch creation logic), 
    # so we have to load hyperparameters for model which were used during cross-validation
    HP = Hyperparameters.from_dict(meta['hypers'])
    LOG(f'Hyperparameters loaded from version={LAUNCH_GOAL.model_version}: {HP._asdict()}')
    fe_model = FeatExtrModel(HP)
    fe_model = fe_model.to(device=CONFIG.cuda_device)
    LOG(f'FE model instance loaded from version={LAUNCH_GOAL.model_version}')
    old_metrics_suite = json.load(io.BytesIO(model_registry.get_asset_content(LAUNCH_GOAL.model_name, LAUNCH_GOAL.model_version, asset_ext='json', asset_classifier='metrics')))
    METRICS_SUITE.update(old_metrics_suite)
    LOG(f'Metrics suite loaded from version={LAUNCH_GOAL.model_version}')
else:
    assert False

## Train

In [106]:
# @launchit.disable_2
train_dataset, test_dataset = create_dataset_couple_for_supervised_training()
data_loader = DataLoader(train_dataset, batch_size=HP.batch_size, shuffle=True)

if HP.refine_fe_model:
    this_fe_model = copy.deepcopy(fe_model)
    this_fe_model.train()
else:
    this_fe_model = fe_model
    this_fe_model.eval()

model = MainModel(train_dataset[0][0].shape, this_fe_model.to(device='cpu'), HP)
model = model.to(device=CONFIG.cuda_device)
loss_fn = nn.CrossEntropyLoss()
lr_params = HP.parse_learn_rate()
optimizer = getattr(torch.optim, HP.optimizer)(model.parameters(), lr=lr_params.learn_rate)
lr_scheduler = LrSchedulerWrapper(optimizer, lr_params)

test_images = test_dataset.datasets[0].to(device=CONFIG.cuda_device)
test_images = einops.rearrange(test_images, 'b h w -> b 1 h w')
test_labels = test_dataset.datasets[1].to(device=CONFIG.cuda_device)

for epoch_ind in tqdm(range(HP.epochs_count), 'Epoch', disable=not CONFIG.is_interactive):
    epoch = epoch_ind + 1
    train_loss_sum, train_loss_denom = 0, 0
    train_accuracy_sum, train_accuracy_denom = 0, 0
    
    for batch in data_loader:
        optimizer.zero_grad()
        
        images = batch[0].to(device=CONFIG.cuda_device)
        images = einops.rearrange(images, 'b h w -> b 1 h w')
        labels = batch[1].to(device=CONFIG.cuda_device)
        logits = model(images)
        loss = loss_fn(logits, labels)
        loss.backward()
        
        optimizer.step()
        
        train_loss_sum += loss.item() * len(batch)
        train_loss_denom += len(batch)
        train_accuracy_sum += (logits.argmax(axis=1) == labels).sum().item() # how many correct predictions
        train_accuracy_denom += len(labels)

    assert train_loss_denom > 0
    assert train_accuracy_denom > 0
    train_loss = train_loss_sum / train_loss_denom
    train_accuracy = train_accuracy_sum / train_accuracy_denom
    summary_writer.add_scalar('train_loss', train_loss, epoch)
    summary_writer.add_scalar('train_accuracy', train_accuracy, epoch)
    METRICS_SUITE['train_loss'].append(train_loss)
    METRICS_SUITE['train_accuracy'].append(train_accuracy)

    lr_scheduler.step(train_loss)

    with torch.no_grad():
        test_logits = model(test_images)
        test_loss = loss_fn(test_logits, test_labels).item()
        predicted_test_labels = test_logits.argmax(axis=1)
        test_accuracy = (predicted_test_labels == test_labels).sum().item() / len(test_labels)
        summary_writer.add_scalar('test_loss', test_loss, epoch)
        summary_writer.add_scalar('test_accuracy', test_accuracy, epoch)
        METRICS_SUITE['test_loss'].append(test_loss)
        METRICS_SUITE['test_accuracy'].append(test_accuracy)
        
    summary_writer.flush()

Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

## Save

In [80]:
# @launchit.disable_2
model_registry = new_model_registry()

with io.BytesIO() as b:
    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'hypers': HP._asdict(),
    }, b)
    model_registry.attach_asset(LAUNCH_GOAL.model_name, LAUNCH_GOAL.model_version, b, asset_ext='pt', asset_classifier='main', replace=True)

with io.StringIO() as b:
    json.dump(METRICS_SUITE, b)
    model_registry.attach_asset(LAUNCH_GOAL.model_name, LAUNCH_GOAL.model_version, b, asset_ext='json', asset_classifier='metrics', replace=True)

# Save Optuna trial result

In [121]:
# @launchit.disable_1
optuna_trial = optuna_multiprocessing.get_trial()

if optuna_trial is not None:
    if 'IS_PRUNED' in optuna_trial.user_attrs:
        raise optuna.exceptions.TrialPruned()

    if not METRICS_SUITE:
        LOG(f'Empty metrics suite. Cancelling model')
        optuna_multiprocessing.save_trial_result(0)
    else:
        match LAUNCH_GOAL.goal:
            case LaunchGoal.TRAIN_MODEL_CV:
                last_std_val_accuracy = METRICS_SUITE['std_val_accuracy'][-1]
                last_mean_val_accuracy = METRICS_SUITE['mean_val_accuracy'][-1]
                    
                if last_std_val_accuracy > 0.05:
                    LOG(f'Unstable condition encountered: {last_mean_val_accuracy=}, {last_std_val_accuracy=}. Cancelling model')
                    optuna_multiprocessing.save_trial_result(0)
                else:
                    optuna_multiprocessing.save_trial_result(last_mean_val_accuracy)
                    LOG(f'Train objective result: {last_mean_val_accuracy=}')
            case LaunchGoal.TRAIN_MODEL:
                # Bad practice due to test data influences model selection
                last_test_accuracy = METRICS_SUITE['test_accuracy'][-1]
                optuna_multiprocessing.save_trial_result(last_test_accuracy)
                LOG(f'Train objective result: {last_test_accuracy=}')
            case _:
                assert False, f'Unsupported {LAUNCH_GOAL.goal=}'

# Launch creation

## TRAIN_MODEL_CV

In [25]:
# @launchit.disable
launchit_t0 = time.time()

In [11]:
# @launchit.disable
launchit_interval = time.time() - launchit_t0

if launchit_interval > 0.05:
    model_version = int(Autoincrement.get(f'{LAUNCH_GOAL.model_group_uri}.{LAUNCH_GOAL.model_name}'))
    assert model_version > 0, model_version
    model_registry_obj = new_model_registry(is_real=True)
    model_registry_obj.register_model(LAUNCH_GOAL.model_name, model_version)
    LOG(f'Model instance registered, version={model_version}')
    
    expandvars = dict(
        PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=LAUNCH_GOAL.model_group_uri,
        MODEL_NAME=LAUNCH_GOAL.model_name,
        MODEL_VERSION=model_version,
        LAUNCH_GOAL=LaunchGoal.TRAIN_MODEL_CV,
    )
    launch_notebook_fname = launchit.launchit(CONFIG.self_fname, launch_serial=model_version, expandvars=expandvars, collect_inds=[1], disable_inds=[2])
    LOG(f'Created launch notebook "{launch_notebook_fname}"')
else:
    LOG('Skip launchit due to mass "Run Cells"')

Model instance registered, version=263
Creating /home/misha/dev/mine/neurovision/denoise/s3_stacked_dae_09-launch263.ipynb
Created launch notebook "/home/misha/dev/mine/neurovision/denoise/s3_stacked_dae_09-launch263.ipynb"


## TRAIN_MODEL

In [31]:
# @launchit.disable
launchit_t0 = time.time()

In [32]:
# @launchit.disable
launchit_interval = time.time() - launchit_t0

if launchit_interval > 0.05:
    model_version = int(Autoincrement.get(f'{LAUNCH_GOAL.model_group_uri}.{LAUNCH_GOAL.model_name}'))
    assert model_version > 0, model_version
    model_registry_obj = new_model_registry(is_real=True)
    model_registry_obj.register_model(LAUNCH_GOAL.model_name, model_version)
    LOG(f'Model instance registered, version={model_version}')
    
    expandvars = dict(
        PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=LAUNCH_GOAL.model_group_uri,
        MODEL_NAME=LAUNCH_GOAL.model_name,
        MODEL_VERSION=model_version,
        LAUNCH_GOAL=LaunchGoal.TRAIN_MODEL,
    )
    launch_notebook_fname = launchit.launchit(CONFIG.self_fname, launch_serial=model_version, expandvars=expandvars, collect_inds=[1], disable_inds=[])
    LOG(f'Created launch notebook "{launch_notebook_fname}"')
else:
    LOG('Skip launchit due to mass "Run Cells"')

Model instance registered, version=135
Creating /home/misha/dev/mine/neurovision/11_cnn/s4_cnn_02-launch135.ipynb
Created launch notebook "/home/misha/dev/mine/neurovision/11_cnn/s4_cnn_02-launch135.ipynb"


## Optuna (model selection)

### Templates

In [28]:
# @launchit.disable
# @launchit.collect_2
optuna_trial = optuna_multiprocessing.get_trial()

if optuna_trial is not None:
    study_serial = optuna_trial.user_attrs['STUDY_SERIAL']
    
    match study_serial:
        case 1:
            HP.random_seed = 42
            HP.images_preprocessing = 'SAMPLE_WISE' # NONE, UNINORM, SAMPLE_WISE, MIN_MAX, STANDARDIZE, ZCA_WHITEN, ZCA_HFR30_WHITEN
            HP.fe_model_units = [
                'conv(1->32(1)x5+bias)->tanh->gain->abs->avg(2)',
                'conv(32->64(16)x5+bias)->tanh->gain->abs->avg(2)'
            ]
            HP.lr_model_units = [
                '200->tanh',
                '10->tanh',
            ]
            HP.refine_fe_model = True
            HP.batch_size = 100
            HP.epochs_count = 15
            HP.optimizer = 'Adam'
            initial_lr = optuna_trial.suggest_float('initial_lr', 0.0005, 0.005)
            plateau_factor = optuna_trial.suggest_float('plateau_factor', 0.1, 0.5)
            plateau_patience = optuna_trial.suggest_int('plateau_patience', HP.epochs_count // 4, HP.epochs_count // 2)
            HP.learn_rate = f'{initial_lr}, plateau(factor={plateau_factor}, patience={plateau_patience})'
            HP.cv_folds_count = 5
        case 2:
            HP.random_seed = 42
            HP.images_preprocessing = 'SAMPLE_WISE' # NONE, UNINORM, SAMPLE_WISE, MIN_MAX, STANDARDIZE, ZCA_WHITEN, ZCA_HFR30_WHITEN
            HP.fe_model_units = [
                'conv(1->32(1)x5+bias)->tanh->gain->abs->avg(2); from:s4_psd_01,450',
                'conv(32->64(16)x5+bias)->tanh->gain->abs->avg(2)'
            ]
            HP.lr_model_units = [
                '200->tanh',
                '10->tanh',
            ]
            HP.refine_fe_model = True
            HP.batch_size = 100
            HP.epochs_count = 15
            HP.optimizer = 'Adam'
            initial_lr = optuna_trial.suggest_float('initial_lr', 0.0001, 0.005)
            plateau_factor = optuna_trial.suggest_float('plateau_factor', 0.1, 0.5)
            plateau_patience = optuna_trial.suggest_int('plateau_patience', HP.epochs_count // 4, HP.epochs_count // 2)
            HP.learn_rate = f'{initial_lr}, plateau(factor={plateau_factor}, patience={plateau_patience})'
            HP.cv_folds_count = 5
        case 3:
            HP.random_seed = 42
            HP.images_preprocessing = 'SAMPLE_WISE' # NONE, UNINORM, SAMPLE_WISE, MIN_MAX, STANDARDIZE, ZCA_WHITEN, ZCA_HFR30_WHITEN
            is_abs1 = optuna_trial.suggest_categorical('is_abs1', [True, False])
            is_abs2 = optuna_trial.suggest_categorical('is_abs2', [True, False])
            lcn_window1 = optuna_trial.suggest_categorical('lcn_window1', [0, 3, 5, 7])
            lcn_window2 = optuna_trial.suggest_categorical('lcn_window2', [0, 3, 5, 7])
            lcn_sigma1 = optuna_trial.suggest_int('lcn_sigma1', 1, 3)
            lcn_sigma2 = optuna_trial.suggest_int('lcn_sigma2', 1, 3)
            inject_abs = lambda flag: 'abs->' if flag else ''
            inject_lcn = lambda w, s: f'LCN({w}, {s})->' if w > 0 else ''
            HP.fe_model_units = [
                f'conv(1->32(1)x5+bias)->tanh->gain->{inject_abs(is_abs1)}{inject_lcn(lcn_window1, lcn_sigma1)}avg(2)',
                f'conv(32->64(16)x5+bias)->tanh->gain->{inject_abs(is_abs2)}{inject_lcn(lcn_window2, lcn_sigma2)}avg(2)',
            ]
            HP.lr_model_units = [
                '200->tanh',
                '10->tanh',
            ]
            HP.refine_fe_model = True
            HP.batch_size = 100
            HP.epochs_count = 15
            HP.optimizer = 'Adam'
            HP.learn_rate = '0.0005, plateau(factor=0.5, patience=5)'
            HP.cv_folds_count = 5
        case _:
            assert False, f'Unsupported {study_serial=}'    

### Unleash

In [29]:
# @launchit.disable
def get_optimize_directions(lg):
    match lg:
        case LaunchGoal.TRAIN_MODEL_CV:
            return ['maximize']
        case LaunchGoal.TRAIN_MODEL:
            return ['maximize']
        case _:
            assert False, f'Unsupported {lg=}'

lg = LaunchGoal.TRAIN_MODEL_CV
expandvars = dict(
    PROJECT_ROOT_PATH=CONFIG.project_root_path,
    MODEL_GROUP_URI=LAUNCH_GOAL.model_group_uri,
    MODEL_NAME=LAUNCH_GOAL.model_name,
    LAUNCH_GOAL=lg.value,
)
study_serial = 3
study_name = f'{CONFIG.self_name}_{expandvars['LAUNCH_GOAL']}_{study_serial}'
rop_task = optuna_multiprocessing.RunOptimizationTask(
    app_name=CONFIG.self_name,
    is_stdout_enabled=False,
    notebook_fname=CONFIG.self_fname,
    notebook_name=CONFIG.self_name,
    model_group_uri=LAUNCH_GOAL.model_group_uri,
    model_name=LAUNCH_GOAL.model_name,
    expandvars=expandvars,
    collect_inds=[2],
    disable_inds=[2],
    run_path=CONFIG.run_path,
    study_serial=study_serial,
    study_name=study_name,
    study_fname=os.path.join(CONFIG.run_path, study_name + '.log'),
    optimize_directions=get_optimize_directions(lg),
)
rop_tasks = [rop_task] * 25
mp_ctx = mp.get_context('spawn') # Req-d for CUDA, fork doesn't work within PyTorch

with mp_ctx.Pool(processes=2, maxtasksperchild=1) as pool:  # maxtasksperchild=1 forces fresh process for each trial to spare resources and avoid possible side effects of processe resue
    pool.map(optuna_multiprocessing.run_optimization, rop_tasks)

[I 2026-02-26 20:13:21,180] A new study created in Journal with name: s4_cnn_02_train_model_cv_3
[I 2026-02-26 20:13:21,181] Using an existing study with name 's4_cnn_02_train_model_cv_3' instead of creating a new one.
[I 2026-02-26 20:22:01,285] Trial 0 finished with value: 0.9907 and parameters: {'is_abs1': True, 'is_abs2': True, 'lcn_window1': 7, 'lcn_window2': 0, 'lcn_sigma1': 2, 'lcn_sigma2': 2}. Best is trial 0 with value: 0.9907.
[I 2026-02-26 20:22:01,368] Using an existing study with name 's4_cnn_02_train_model_cv_3' instead of creating a new one.
[I 2026-02-26 20:22:59,137] Trial 1 finished with value: 0.9901833333333332 and parameters: {'is_abs1': False, 'is_abs2': False, 'lcn_window1': 7, 'lcn_window2': 3, 'lcn_sigma1': 1, 'lcn_sigma2': 2}. Best is trial 0 with value: 0.9907.
[I 2026-02-26 20:22:59,223] Using an existing study with name 's4_cnn_02_train_model_cv_3' instead of creating a new one.
[I 2026-02-26 20:28:49,649] Trial 3 finished with value: 0.9911333333333333 and

In [30]:
# @launchit.disable
study = optuna.create_study(
    study_name=rop_task.study_name,
    storage=JournalStorage(JournalFileBackend(file_path=rop_task.study_fname)),
    load_if_exists=True, 
)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

LOG('Study statistics: ')
LOG(f'\tNumber of finished trials: {len(study.trials)}')
LOG(f'\tNumber of pruned trials: {len(pruned_trials)}')
LOG(f'\tNumber of complete trials: {len(complete_trials)}')

if len(study.directions) == 1:
    LOG('Best trial:')
    trial = study.best_trial
    
    LOG(f'\tValue: {trial.value}')
    LOG(f'\tModel version: {trial.user_attrs['MODEL_VERSION']}')
    
    LOG('  Params: ')
    for key, value in trial.params.items():
        LOG(f'\t\t{key}: {value}')
else:
    print(f"Number of trials on the Pareto front: {len(study.best_trials)}")

    for i in range(3):
        print(f"Trial with lowest loss_{i}:")
        trial = min(study.best_trials, key=lambda t: t.values[i])
        print(f"\tnumber: {trial.number}")
        print(f"\tmver: {trial.user_attrs['MODEL_VERSION']}")
        print(f"\tparams: {trial.params}")
        print(f"\tvalues: {trial.values}")

[I 2026-02-26 21:50:34,017] Using an existing study with name 's4_cnn_02_train_model_cv_3' instead of creating a new one.


Study statistics: 
	Number of finished trials: 25
	Number of pruned trials: 7
	Number of complete trials: 18
Best trial:
	Value: 0.9925333333333335
	Model version: 125
  Params: 
		is_abs1: True
		is_abs2: True
		lcn_window1: 3
		lcn_window2: 7
		lcn_sigma1: 2
		lcn_sigma2: 1
